# 12 — Reprendre un entraînement PPO

Charge un checkpoint enregistré par le notebook PPO. Aucun entraînement ne démarre à l’ouverture. Les états internes et la politique chargée doivent correspondre exactement au même espace d’observation.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'OUARTI'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier OUARTI absent : extraire OUARTI a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


In [ ]:
import socket, json, ast, time
HOST='127.0.0.1'
PORT_UDP=2345
PORT_TCP=2347

def management(message,timeout=3):
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout);s.sendall(message.encode('ascii'))
        data=b''
        end=time.monotonic()+timeout
        while time.monotonic()<end and len(data)<1024*1024:
            try:chunk=s.recv(65536)
            except socket.timeout:break
            if not chunk:break
            data+=chunk
            try:return json.loads(data)
            except (json.JSONDecodeError, UnicodeDecodeError):pass
            try:return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeDecodeError):pass
        if data:
            raise RuntimeError('Reponse TCP incomplete : '+repr(data[:160]))
    raise RuntimeError('Pas de reponse sur le port management')

def open_telemetry(timeout=4):
    s=socket.socket(socket.AF_INET,socket.SOCK_DGRAM);s.settimeout(.4)
    s.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        s.send(b'connect')
        try:response=s.recv(2**18)
        except socket.timeout:continue
        if response==b'identified':return s
    s.close()
    raise TimeoutError('Pas de confirmation UDP sur 2345 : verifier jeu en piste et plugin sensors_par.')

def next_packet(s,timeout=3):
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        try:message=s.recv(2**18)
        except socket.timeout:continue
        if message==b'disconnect':raise ConnectionError('Le jeu a ferme le flux UDP')
        if message==b'identified':continue
        raw=json.loads(message)
        if not isinstance(raw,dict):continue
        raw['_capture_wall_time']=time.time()
        return raw
    raise TimeoutError('Aucun echantillon recu.')

def close_telemetry(s):
    if s is not None:
        try:s.send(b'disconnect')
        except OSError:pass
        s.close()


In [ ]:
import os, sys, json, socket, ast, time, numpy as np, pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from scipy.spatial import cKDTree
from pathlib import Path

# Conditions pour les commandes via le plugin : vjoy_executed_by_server=True,
# sampling_freq=100, plateforme Windows, voiture/piste exactement celles attendues.
TRACK_ATTENDU='imola'     # a adapter a ta session
VOITURE_ATTENDUE='abarth500'  # a adapter selon CarName du plugin
TOTAL_TIMESTEPS=2048
ACTIVER_ENTRAINEMENT_REEL=False

def check_session():
    plugin=management('get_config');sess=management('get_static_info')
    name=sess.get('TrackName','')
    if sess.get('TrackConfiguration'): name+='-'+sess['TrackConfiguration']
    if name!=TRACK_ATTENDU or sess.get('CarName')!=VOITURE_ATTENDUE:
        raise RuntimeError('Session differente : %s/%s. Attendu : %s/%s'%(name,sess.get('CarName'),TRACK_ATTENDU,VOITURE_ATTENDUE))
    if not plugin.get('vjoy_executed_by_server',False):
        raise RuntimeError('Ce notebook PPO exige vjoy_executed_by_server=True dans sensors_par/config.py (en jeu).')
    if plugin.get('sampling_freq')!=100:
        raise RuntimeError('Avec vJoy serveur, sampling_freq doit etre 100 Hz dans le plugin.')
    if plugin.get('ego_sampling_freq',0)<=0:
        raise RuntimeError('ego_sampling_freq invalide.')
    if not sess.get('autoShifterOn',False):
        raise RuntimeError('Activer la boite automatique dans Assetto Corsa.')
    return plugin,sess

class Track:
    def __init__(self,filename):
        frame=pd.read_csv(filename)
        for col in ['left_border_x','left_border_y','right_border_x','right_border_y']:
            if col not in frame:raise ValueError('Carte circuit manquante : '+col)
        L=frame[['left_border_x','left_border_y']].values
        R=frame[['right_border_x','right_border_y']].values
        self.center=(L+R)/2.
        self.widths=np.linalg.norm(L-R,axis=1)
        self.delta=np.roll(self.center,-1,axis=0)-self.center
        self.segment_lengths=np.linalg.norm(self.delta,axis=1)
        self.arc=np.r_[0,np.cumsum(self.segment_lengths)]
        self.total=float(self.arc[-1])
        self.tree=cKDTree(self.center)
    def project(self,xy):
        # approx nearest point adequate for a first prototype, not final racing line geometry.
        _,idx=self.tree.query(xy)
        idx=int(idx)
        hd=np.arctan2(self.delta[idx,1],self.delta[idx,0])
        u=self.delta[idx]/max(self.segment_lengths[idx],1e-8)
        lateral=float(np.cross(u,xy-self.center[idx]))
        return self.arc[idx],lateral,float(self.widths[idx]),float(hd)

def obs(raw,track):
    # Observation 9 values; no global position is passed to policy.
    xy=np.array([float(raw['world_position_x']),float(raw['world_position_y'])])
    arc,lateral,width,heading=track.project(xy)
    ang=((float(raw['yaw'])-heading+np.pi)%(2*np.pi))-np.pi
    state=[float(raw.get('speed',0))/100.,float(raw.get('accelX',0))/3.,
           float(raw.get('accelY',0))/3.,float(raw.get('angular_velocity_y',0))/4.,
           np.clip(lateral/max(width/2,1),-2,2),np.sin(ang),np.cos(ang),
           float(raw.get('accStatus',0)),float(raw.get('brakeStatus',0))]
    return np.clip(np.asarray(state,dtype=np.float32),-5,5),arc,lateral,width

class LiveAssettoEnv(gym.Env):
    metadata={'render_modes':[]}
    def __init__(self,track,episode_seconds=30):
        super().__init__()
        self.track=track
        self.action_space=spaces.Box(low=np.array([-1.,0.,0.],dtype=np.float32),high=np.ones(3,dtype=np.float32))
        self.observation_space=spaces.Box(-5,5,shape=(9,),dtype=np.float32)
        self.episode_seconds=episode_seconds;self.sock=None;self.raw=None
        self.t0=0;self.arc=0;self.last_damage=0;self.last_obs=np.zeros(9,np.float32)
    def hold(self):
        if self.sock and self.raw:
            command={'steer':0.,'acc':-1.,'brake':1.,'enable_gear_shift':False,
                     'shift_up':False,'shift_down':False,'server_steps':int(self.raw.get('steps',0)),
                     'ouarti_watchdog':True}
            try:self.sock.send(json.dumps(command).encode('utf8'))
            except OSError:pass
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        self.close()
        management('reset')
        time.sleep(1)
        self.sock=open_telemetry()
        self.raw=next_packet(self.sock)
        self.last_obs,self.arc,_,_=obs(self.raw,self.track)
        self.last_damage=float(self.raw.get('damage_total',0))
        self.t0=time.monotonic()
        return self.last_obs.copy(),{}
    def step(self,action):
        if self.sock is None:raise RuntimeError('Appeler reset() avant step().')
        action=np.asarray(action,dtype=np.float32)
        if action.shape!=(3,) or not np.isfinite(action).all():raise ValueError('Action non finie ou taille invalide.')
        steer=float(np.clip(action[0],-1,1));acc=float(np.clip(action[1],0,1));brake=float(np.clip(action[2],0,1))
        command=dict(steer=steer,acc=2*acc-1,brake=2*brake-1,
                     enable_gear_shift=False,shift_up=False,shift_down=False,
                     server_steps=int(self.raw.get('steps',0)),ouarti_watchdog=True)
        self.sock.send(json.dumps(command).encode('utf8'))
        try:
            new=next_packet(self.sock,timeout=2)
            state,arc,lateral,width=obs(new,self.track)
            move=(arc-self.arc+self.track.total/2)%self.track.total-self.track.total/2
            dt=float(new.get('timestamp_unix',time.time()))-float(self.raw.get('timestamp_unix',time.time()))
            if dt<=0 or dt>.35:raise RuntimeError('Timestamps incoherents.')
            # Ignore teleports and punishes severe errors. Starting prototype, not validated racing policy.
            if abs(move)>max(3,120*dt*1.5):move=0
            damage=float(new.get('damage_total',0))
            collision=damage>self.last_damage+.01
            tyres=int(new.get('numberOfTyresOut',0))
            offtrack=tyres>2 or abs(lateral)>width/2
            reward=float(move-0.1*dt-25*collision-15*offtrack)
            terminated=bool(collision or offtrack)
            truncated=time.monotonic()-self.t0>=self.episode_seconds
            self.raw=new;self.arc=arc;self.last_obs=state;self.last_damage=damage
            if terminated or truncated:self.hold()
            return state,reward,terminated,truncated,{'speed':float(new.get('speed',0)),
                     'collision':collision,'offtrack':offtrack,'progress_m':move}
        except (OSError,ValueError,RuntimeError,ConnectionError,TimeoutError) as exc:
            self.hold()
            return self.last_obs.copy(),-50.,False,True,{'sensor_error':str(exc)}
    def close(self):
        if self.sock:
            self.hold()
            close_telemetry(self.sock)
            self.sock=None

class HoldCallback(BaseCallback):
    def _on_step(self):
        return True
    def _on_rollout_end(self):
        # Stop any acceleration while PPO is updating neural weights.
        self.training_env.env_method('hold')


In [ ]:
CHEMIN_MODELE=PROJECT/'models'/'ppo_assetto_prototype.zip'
ACTIVER_REPRISE=False
TIMESTEPS_SUPPLEMENTAIRES=1024
if not ACTIVER_REPRISE:
    print('Reprise desactivee :',CHEMIN_MODELE)
else:
    if not CHEMIN_MODELE.exists():raise FileNotFoundError(str(CHEMIN_MODELE))
    check_session()
    circuit=Track(GYM_ROOT/'AssettoCorsaConfigs'/'tracks'/(TRACK_ATTENDU+'.csv'))
    env=LiveAssettoEnv(circuit)
    try:
        agent=PPO.load(str(CHEMIN_MODELE),env=env,device='auto')
        agent.learn(total_timesteps=TIMESTEPS_SUPPLEMENTAIRES,reset_num_timesteps=False,callback=HoldCallback())
        agent.save(str(PROJECT/'models'/'ppo_assetto_repris'))
    finally:
        env.close()
